# 03 — Feature Engineering
**Owner:** All members (coordinate before editing)  |  **Phase:** 2  |  **Date:** May 12

Objectives: date features, financial ratios, Zimbabwe-context features, interaction features.

In [ ]:
import pandas as pd
import sys, pathlib
sys.path.insert(0, str(pathlib.Path(".").resolve()))

from src.data_loader import load_train, load_test
from src.feature_engineering import (
    extract_date_features, build_ratio_features,
    build_zimbabwe_features, build_interaction_features,
    add_missing_flags, engineer_all_features
)

train = load_train()
test  = load_test()

## 1. Date Features

In [ ]:
train = extract_date_features(train)
date_feats = ["client_age_at_approval","disburse_lag_days",
              "time_to_first_payment_days","loan_duration_days","approval_month","approval_quarter"]
print(train[date_feats].describe().T.round(2))

## 2. Financial Ratio Features

In [ ]:
train = build_ratio_features(train)
ratio_feats = ["debt_to_income","total_interest_cost","loan_to_income_ratio",
               "income_per_dependent","obligation_burden"]
print(train[ratio_feats].describe().T.round(4))
print("\nCorrelation with Target:")
print(train[ratio_feats + ["Target"]].corr()["Target"].sort_values())

## 3. Zimbabwe-Specific Features

In [ ]:
train = build_zimbabwe_features(train)
zim_feats = ["urban_province","informal_sector_flag","productive_loan",
              "high_rate_mfi_loan","salary_backed_loan","has_collateral"]
for feat in zim_feats:
    rate = train.groupby(feat)["Target"].mean()
    print(f"{feat}:\n{rate}\n")

## 4. Interaction Features

In [ ]:
train = build_interaction_features(train)
print(train[["province_x_sector","product_x_purpose","rate_x_obligations"]].head(10))

## 5. Missing Indicator Flags

In [ ]:
train = add_missing_flags(train)
flag_cols = [c for c in train.columns if c.endswith("_missing")]
print("Flags added:", flag_cols)
print(train[flag_cols].sum())

## 6. Correlation of New Features with Target

In [ ]:
new_num = ["debt_to_income","total_interest_cost","loan_to_income_ratio",
           "income_per_dependent","obligation_burden","rate_x_term",
           "client_age_at_approval","disburse_lag_days","time_to_first_payment_days"]
corr = train[new_num + ["Target"]].corr()["Target"].drop("Target").sort_values()
print(corr.to_string())